In [ ]:
from google.colab import drive
drive.mount('/content/drive')

버스정류장 전처리

In [ ]:
import pandas as pd

# 1. 데이터 불러오기 (인코딩 에러 방지)
file_path = "/content/drive/MyDrive/국토교통부_전국 버스정류장 위치정보_20251031.csv"
try:
    df_bus = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df_bus = pd.read_csv(file_path, encoding='cp949')

# 컬럼 이름 변경: '위도', '경도', '모바일단축번호'를 올바른 이름으로 매핑
# 실제 파일의 헤더가 없거나 잘못 파싱된 경우 발생할 수 있습니다.
# 커널 상태의 df_bus 정보를 바탕으로 컬럼을 재정의합니다.
# 원본 데이터에 따라 아래 컬럼 이름은 달라질 수 있습니다.
df_bus = df_bus.rename(columns={
    '35.580498': '위도', # 위도 데이터가 있는 실제 컬럼 이름
    '127.065887': '경도', # 경도 데이터가 있는 실제 컬럼 이름
    '3090600': '모바일단축번호' # 모바일단축번호 데이터가 있는 실제 컬럼 이름
})

# 2. 필수 데이터 결측치 제거: 위도 또는 경도 정보가 없는 정류장 삭제 (5건)
df_bus = df_bus.dropna(subset=['위도', '경도'])

# 3. 서식 정제 및 결측치 처리: '모바일단축번호'
# 결측치는 빈칸으로 처리하고, 소수점 형태(예: 540001.0)를 깔끔한 문자열(예: '540001')로 변환
df_bus['모바일단축번호'] = df_bus['모바일단축번호'].fillna(-1).astype(int).astype(str).replace('-1', '')

# 4. 이상치 제거: 대한민국의 실제 위경도 범위를 벗어난 좌표 오류 데이터 삭제 (7건)
# (위경도가 바뀌어 입력되었거나 오타가 발생한 경우)
invalid_bus = df_bus[
    (df_bus['위도'] < 33) | (df_bus['위도'] > 39) |
    (df_bus['경도'] < 124) | (df_bus['경도'] > 132)
]
df_bus_clean = df_bus.drop(index=invalid_bus.index)

# 5. 전처리 완료된 데이터 저장
df_bus_clean.to_csv("preprocessed_버스정류장_v3.csv", index=False, encoding='utf-8-sig')

print("버스정류장 데이터 전처리 완료 및 저장 성공 (preprocessed_버스정류장_v3.csv)")

버스정류장 데이터 전처리 완료 및 저장 성공 (preprocessed_버스정류장_v3.csv)


주거용 건물 전처리

In [ ]:
import pandas as pd

# ── 1. 원본 파일 로드 ──────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/03. 표제부_20260413190447.csv', low_memory=False, encoding='utf-8')

# ── 2. 주거용 필터링 (단독주택 + 공동주택) ───────────────────
df = df[df['주용도코드명'].isin(['단독주택', '공동주택'])].reset_index(drop=True)

# ── 3. 필요 컬럼만 선택 ───────────────────────────────────────
keep_cols = [
    '대지위치',
    '도로명대지위치',
    '건물명',
    '주용도코드명',
    '지상층수',
    '세대수(세대)',
    '가구수(가구)',
    '연면적(㎡)',
]
df = df[keep_cols].copy()

# ── 4. 세대수 + 가구수 → 수요 합산 후 원본 컬럼 삭제 ─────────
df['수요(세대+가구)'] = df['세대수(세대)'].fillna(0).astype(int) + df['가구수(가구)'].fillna(0).astype(int)
df = df.drop(columns=['세대수(세대)', '가구수(가구)'])

# ── 5. 컬럼 순서 정리 ─────────────────────────────────────────
df = df[['대지위치', '도로명대지위치', '건물명', '주용도코드명', '지상층수', '수요(세대+가구)', '연면적(㎡)']]

# ── 6. 확인 ───────────────────────────────────────────────────
print(f"행수: {len(df):,}")
print(f"컬럼: {df.columns.tolist()}")
df.head()

df.to_csv('/content/drive/MyDrive/주거용건물최종.csv', index=False, encoding='cp949')

행수: 26,671
컬럼: ['대지위치', '도로명대지위치', '건물명', '주용도코드명', '지상층수', '수요(세대+가구)', '연면적(㎡)']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import urllib.request
import urllib.parse
import json
import time
import os
from tqdm import tqdm

KAKAO_KEY   = "23ef9b3d396dee6c4834b2da300c921d"
INPUT_FILE  = "/content/drive/MyDrive/주거용건물최종.csv"
OUTPUT_FILE = "/content/drive/MyDrive/주거용건물최종_좌표.csv"
DELAY_SEC   = 0.05

def geocode(address):
    if not address or str(address).strip() == "nan":
        return None, None
    address = address.replace("번지", "").strip()
    try:
        query = urllib.parse.urlencode({"query": address})
        url = "https://dapi.kakao.com/v2/local/search/address.json?" + query
        req = urllib.request.Request(url, headers={"Authorization": "KakaoAK " + KAKAO_KEY})
        res = urllib.request.urlopen(req, timeout=5)
        data = json.loads(res.read().decode("utf-8"))
        if data.get("documents"):
            doc = data["documents"][0]
            return float(doc["y"]), float(doc["x"])
    except Exception as e:
        print("  오류: " + address + " -> " + str(e))
    return None, None

def main():
    # Changed encoding from 'utf-8-sig' to 'cp949' as a common fix for Korean CSV files
    df = pd.read_csv(INPUT_FILE, encoding="cp949")
    print("총 " + str(len(df)) + "행 로드 완료")

    if os.path.exists(OUTPUT_FILE):
        done = pd.read_csv(OUTPUT_FILE, encoding="cp949")
        start_idx = len(done)
        print("이어서 진행: " + str(start_idx) + "행 이미 완료")
    else:
        done = df.iloc[:0].copy()
        done["위도"] = None
        done["경도"] = None
        start_idx = 0

    results = []
    for i, row in tqdm(df.iloc[start_idx:].iterrows(), total=len(df)-start_idx, desc="지오코딩"):
        addr = row.get("도로명대지위치") if pd.notna(row.get("도로명대지위치")) else row.get("대지위치")
        lat, lon = geocode(str(addr))
        r = row.to_dict()
        r["위도"] = lat
        r["경도"] = lon
        results.append(r)

        if len(results) % 100 == 0:
            tmp = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
            tmp.to_csv(OUTPUT_FILE, index=False, encoding="cp949") # Changed encoding here too

        time.sleep(DELAY_SEC)

    final = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
    final_clean = final.dropna(subset=["위도", "경도"])
    final_clean.to_csv(OUTPUT_FILE, index=False, encoding="cp949") # Changed encoding here too
    print("최종 완료! 총 " + str(len(final)) + "행 / 좌표 확보 " + str(len(final_clean)) + "개 (" + str(round(len(final_clean)/len(final)*100,1)) + "%)")

main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
총 26671행 로드 완료


지오코딩:   0%|          | 99/26671 [01:05<4:50:31,  1.52it/s]/tmp/ipykernel_528/503048468.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
지오코딩: 100%|██████████| 26671/26671 [4:54:37<00:00,  1.51it/s]
/tmp/ipykernel_528/503048468.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final = pd.concat([done, pd.DataFrame(results)], ignore_index=True)


최종 완료! 총 26671행 / 좌표 확보 23918개 (89.7%)


In [ ]:
가중치 셀 추가

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

gdf_grid = gpd.read_file("vl_blk.shp", encoding="cp949")
df = pd.read_csv("/content/drive/MyDrive/주거용건물최종_좌표.csv", encoding="cp949")

gdf_bldg = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["경도"], df["위도"]), crs="EPSG:4326")
gdf_bldg = gdf_bldg.to_crs("EPSG:5179")

joined = gpd.sjoin(gdf_bldg, gdf_grid[["gid", "val", "geometry"]], how="left", predicate="within")
joined = joined.rename(columns={"val": "격자인구"})
joined = joined.drop(columns=["index_right"])

# 1. capacity
joined["capacity"] = np.where(
    joined["수요(세대+가구)"] > 0,
    joined["수요(세대+가구)"],
    np.log1p(joined["연면적(㎡)"])
)

# 2. 비율
grid_cap = joined.groupby("gid")["capacity"].sum().reset_index()
grid_cap.columns = ["gid", "격자capacity합계"]
joined = joined.merge(grid_cap, on="gid", how="left")
joined["비율"] = joined["capacity"] / joined["격자capacity합계"]

# 3. raw_weight
joined["raw_weight"] = (joined["격자인구"] * joined["비율"]).fillna(0)

# 4. final_weight
joined["final_weight"] = np.log1p(joined["raw_weight"])

# final_weight만 원래 df에 추가
df["final_weight"] = joined["final_weight"].values
df.to_csv("/content/drive/MyDrive/주거용건물최종_좌표.csv", index=False, encoding="utf-8-sig")

print("완료! 총 " + str(len(df)) + "행")
print(df[["대지위치", "수요(세대+가구)", "연면적(㎡)", "위도", "경도", "final_weight"]].head(10))

구르미 기존 거점 7개 카카오 지오코딩 (좌표 추출)

In [ ]:
import pandas as pd
import urllib.request
import urllib.parse
import json
from google.colab import drive

drive.mount('/content/drive')

KAKAO_KEY = "23ef9b3d396dee6c4834b2da300c921d"

data = [
    {"터미널": "영주 자전거 공원", "총 거치대 수량(개)": 21, "위치": "영주동 511-1번지"},
    {"터미널": "무섬 마을",       "총 거치대 수량(개)": 20, "위치": "수도리 268-1번지"},
    {"터미널": "한정교",          "총 거치대 수량(개)": 10, "위치": "문정동 706번지"},
    {"터미널": "서천주차장",      "총 거치대 수량(개)": 14, "위치": "가흥동 1408번지"},
    {"터미널": "선비촌",          "총 거치대 수량(개)": 15, "위치": "소백로 2796번지"},
    {"터미널": "용혈폭포",        "총 거치대 수량(개)": 10, "위치": "용혈리 1344-1번지"},
    {"터미널": "술바위교차로",    "총 거치대 수량(개)": 10, "위치": "휴천동 1213-18번지"},
]

def geocode(address):
    address = ("경상북도 영주시 " + address).replace("번지", "").strip()
    try:
        query = urllib.parse.urlencode({"query": address})
        url = "https://dapi.kakao.com/v2/local/search/address.json?" + query
        req = urllib.request.Request(url, headers={"Authorization": "KakaoAK " + KAKAO_KEY})
        res = urllib.request.urlopen(req, timeout=5)
        result = json.loads(res.read().decode("utf-8"))
        if result.get("documents"):
            doc = result["documents"][0]
            return float(doc["y"]), float(doc["x"])
    except Exception as e:
        print("오류:", address, e)
    return None, None

for row in data:
    lat, lon = geocode(row["위치"])
    row["위도"] = lat
    row["경도"] = lon
    print(f"{row['터미널']}: {lat}, {lon}")

df = pd.DataFrame(data)
df.to_csv("/content/drive/MyDrive/구르미_터미널_좌표.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료!")
print(df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
영주 자전거 공원: 36.8246790791725, 128.61585862526
무섬 마을: 36.7331590343526, 128.621034728972
한정교: 36.8003509948257, 128.614905949604
서천주차장: 36.8217430074986, 128.613500655412
선비촌: 36.9284735311732, 128.58343367327
용혈폭포: 36.718587745264, 128.653782582992
술바위교차로: 36.8180154370523, 128.637134542705

저장 완료!
         터미널  총 거치대 수량(개)             위치         위도          경도
0  영주 자전거 공원           21    영주동 511-1번지  36.824679  128.615859
1      무섬 마을           20    수도리 268-1번지  36.733159  128.621035
2        한정교           10      문정동 706번지  36.800351  128.614906
3      서천주차장           14     가흥동 1408번지  36.821743  128.613501
4        선비촌           15     소백로 2796번지  36.928474  128.583434
5       용혈폭포           10   용혈리 1344-1번지  36.718588  128.653783
6     술바위교차로           10  휴천동 1213-18번지  36.818015  128.637135


최종 시각화 코드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import folium
from pyproj import Transformer
from sklearn.metrics.pairwise import euclidean_distances

# ===== 1. 데이터 로드 =====
df_filtered = pd.read_csv("/content/drive/MyDrive/건물_클러스터결과.csv", encoding="utf-8-sig")
df_deadzone = pd.read_csv("/content/drive/MyDrive/탈락건물.csv", encoding="utf-8-sig")
df_deadzone_points = pd.read_csv("/content/drive/MyDrive/데드존포인트.csv", encoding="utf-8-sig")
bus_raw = pd.read_csv("/content/drive/MyDrive/preprocessed_버스정류장_v3.csv", encoding="cp949")
bus = bus_raw[bus_raw["도시명"].str.contains("영주", na=False)].reset_index(drop=True)

gurumi = pd.DataFrame([
    {"터미널": "영주 자전거 공원", "위도": 36.824679, "경도": 128.615859},
    {"터미널": "무섬 마을",        "위도": 36.733159, "경도": 128.621035},
    {"터미널": "한정교",           "위도": 36.800351, "경도": 128.614906},
    {"터미널": "서천주차장",       "위도": 36.821743, "경도": 128.613501},
    {"터미널": "선비촌",           "위도": 36.928474, "경도": 128.583434},
    {"터미널": "용혈폭포",         "위도": 36.718588, "경도": 128.653783},
    {"터미널": "술바위교차로",     "위도": 36.818015, "경도": 128.637135},
])

stations = pd.DataFrame([
    {"이름": "영주역", "위도": 36.8100807321233,  "경도": 128.625043239946},
    {"이름": "풍기역", "위도": 36.8737645479829,  "경도": 128.524738278245},
])

INDUSTRIAL_ZONE_COORDS = [
    [36.7798, 128.6197], [36.7751, 128.6167], [36.7808, 128.6076],
    [36.7895, 128.6181], [36.7847, 128.6248],
]
INDUSTRIAL_LAT = np.mean([c[0] for c in INDUSTRIAL_ZONE_COORDS])
INDUSTRIAL_LON = np.mean([c[1] for c in INDUSTRIAL_ZONE_COORDS])

Shuttle_station = [
    [36.8104, 128.6283],  # 영동삼거리
    [36.8178, 128.6276],  # 샛빛학원
    [36.8231, 128.6280],  # 하동물병원 건너편
    [36.8233, 128.6356],  # 청구아파트앞
    [36.8274, 128.6311],  # 동산교회
    [36.8272, 128.6194],  # 영광중학교건너
    [36.8197, 128.6199],  # 영주시산림조합앞
    [36.8146, 128.6190],  # 꽃동산
    [36.8102, 128.6177],  # 참깨나라앞
    [36.8075, 128.6267],  # 목민로21커피앞
    [36.8109, 128.6078],  # 센텀파크.피카소
    [36.8183, 128.6046],  # 부영아파트
    [36.8308, 128.6078],  # 제일고등학교정문
    [36.8674, 128.5253],  # 백산그랜드아파트앞
    [36.8693, 128.5323],  # 풍기중학교정문
    [36.8731, 128.5242],  # 풍기역버스정류장
]

new_purple_points = [
    {"name": "후보지 1", "lat": 36.80252145383021, "lon": 128.60323328038234},
    {"name": "후보지 2", "lat": 36.7803194648999,  "lon": 128.63212299847297},
]

# ===== 2. UTM 변환 =====
to_utm_tr = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
to_wgs_tr = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)

def utm(lat, lon):
    x, y = to_utm_tr.transform(lon, lat)
    return x, y

def wgs(x, y):
    lon, lat = to_wgs_tr.transform(x, y)
    return lat, lon

bus["UTM_X"], bus["UTM_Y"] = zip(*bus.apply(lambda r: utm(r["위도"], r["경도"]), axis=1))
gurumi["UTM_X"], gurumi["UTM_Y"] = zip(*gurumi.apply(lambda r: utm(r["위도"], r["경도"]), axis=1))
stations["UTM_X"], stations["UTM_Y"] = zip(*stations.apply(lambda r: utm(r["위도"], r["경도"]), axis=1))
bus_utm = bus[["UTM_X", "UTM_Y"]].values
ind_utm_x, ind_utm_y = utm(INDUSTRIAL_LAT, INDUSTRIAL_LON)

# ===== 3. 중복 제거 함수 =====
def remove_duplicates(candidates_df, min_dist=300):
    if len(candidates_df) == 0:
        return candidates_df
    coords = np.array([utm(r["위도"], r["경도"]) for _, r in candidates_df.iterrows()])
    used = [False] * len(candidates_df)
    result = []
    for i in range(len(candidates_df)):
        if used[i]: continue
        result.append(candidates_df.iloc[i])
        for j in range(i+1, len(candidates_df)):
            if not used[j]:
                if np.linalg.norm(coords[i] - coords[j]) < min_dist:
                    used[j] = True
    return pd.DataFrame(result).reset_index(drop=True)

# ===== 4. 출발지 - 셔틀 이용자 =====
selected_shuttle = [
    {"정류장명": "영광중학교건너", "위도": 36.8272, "경도": 128.6194, "유형": "출발지_셔틀이용자", "기준": "셔틀클러스터", "거리_m": 0, "기준_가중치": 970.3},
    {"정류장명": "샛빛학원",       "위도": 36.8178, "경도": 128.6276, "유형": "출발지_셔틀이용자", "기준": "셔틀클러스터", "거리_m": 0, "기준_가중치": 930.0},
    {"정류장명": "센텀파크.피카소","위도": 36.8109, "경도": 128.6078, "유형": "출발지_셔틀이용자", "기준": "셔틀클러스터", "거리_m": 0, "기준_가중치": 744.0},
    {"정류장명": "영동삼거리",     "위도": 36.8104, "경도": 128.6283, "유형": "출발지_셔틀이용자", "기준": "셔틀클러스터", "거리_m": 0, "기준_가중치": 704.9},
]
df_dep_shuttle = pd.DataFrame(selected_shuttle)
print(f"\n[출발지 - 셔틀 이용자] {len(df_dep_shuttle)}개")
print(df_dep_shuttle[["정류장명", "위도", "경도", "기준_가중치"]].to_string())

# ===== 5. 출발지 - 데드존 이용자 =====
deadzone_utm = df_deadzone[["UTM_X", "UTM_Y"]].values

departure_deadzone = []
for _, b_row in bus.iterrows():
    b_coord = np.array([[b_row["UTM_X"], b_row["UTM_Y"]]])
    dists = euclidean_distances(b_coord, deadzone_utm).flatten()
    nearby = (dists <= 500).sum()
    dist_to_ind = euclidean_distances([[ind_utm_x, ind_utm_y]], b_coord).flatten()[0]
    if nearby >= 40 and dist_to_ind <= 10000:
        departure_deadzone.append({
            "유형": "출발지_데드존이용자", "기준": "데드존밀집지",
            "정류장명": b_row["정류장명"], "위도": b_row["위도"], "경도": b_row["경도"],
            "근처건물수": int(nearby), "거리_m": 0, "기준_가중치": 0
        })

df_dep_deadzone = remove_duplicates(
    pd.DataFrame(departure_deadzone)
    .sort_values("근처건물수", ascending=False)
    .drop_duplicates(subset=["정류장명"]).reset_index(drop=True)
)

# 한국폴리텍대학, 거리마을 제거
df_dep_deadzone = df_dep_deadzone[
    ~df_dep_deadzone["정류장명"].isin(["한국폴리텍대학", "거리마을"])
].reset_index(drop=True)
print(f"\n[출발지 - 데드존 이용자] {len(df_dep_deadzone)}개")

# ===== 5-1. 도착지 - 역 연계 =====
arrival_station = []
for _, st in stations.iterrows():
    coord = np.array([[st["UTM_X"], st["UTM_Y"]]])
    dists = euclidean_distances(coord, bus_utm).flatten()
    nearest_idx = dists.argmin()
    nearest_dist = dists[nearest_idx]
    if nearest_dist <= 500:
        b = bus.iloc[nearest_idx]
        arrival_station.append({
            "정류장명": b["정류장명"],
            "위도": b["위도"],
            "경도": b["경도"],
            "기준": st["이름"]
        })
    else:
        arrival_station.append({
            "정류장명": st["이름"],
            "위도": st["위도"],
            "경도": st["경도"],
            "기준": st["이름"]
        })
df_arr_station = pd.DataFrame(arrival_station)
print(f"\n[도착지 - 역 연계] {len(df_arr_station)}개")
print(df_arr_station.to_string())

# ===== 6. 커버리지 평가 =====
building_utm     = df_filtered[["UTM_X", "UTM_Y"]].values
building_weights = df_filtered["가중치"].values
total_weight     = building_weights.sum()

def calc_coverage(coords_utm, radius=500):
    covered = np.zeros(len(building_utm), dtype=bool)
    for coord in coords_utm:
        dists = euclidean_distances([coord], building_utm).flatten()
        covered |= (dists <= radius)
    w = building_weights[covered].sum()
    return w, w / total_weight * 100

gurumi_utm = gurumi[["UTM_X", "UTM_Y"]].values
w_before, pct_before = calc_coverage(gurumi_utm)

new_coords = []
for _, r in df_dep_shuttle.iterrows():  new_coords.append(utm(r["위도"], r["경도"]))
for _, r in df_dep_deadzone.iterrows(): new_coords.append(utm(r["위도"], r["경도"]))
for _, r in df_arr_station.iterrows():  new_coords.append(utm(r["위도"], r["경도"]))
for p in new_purple_points:             new_coords.append(utm(p["lat"], p["lon"]))

all_utm = np.vstack([gurumi_utm, np.array(new_coords)])
w_after, pct_after = calc_coverage(all_utm)

print("\n" + "="*50)
print(f"  기존 구르미 7개소         : {pct_before:.1f}%")
print(f"  신규 후보지 추가 후       : {pct_after:.1f}%")
print(f"  커버리지 증가             : +{pct_after - pct_before:.1f}%p")
print("="*50)

# ===== 7. 시각화 =====
m = folium.Map(location=[36.82, 128.62], zoom_start=12)

folium.Polygon(locations=INDUSTRIAL_ZONE_COORDS, color="green",
               fill=True, fill_opacity=0.2, tooltip="산단 지역").add_to(m)

for _, r in gurumi.iterrows():
    folium.CircleMarker(
        location=[r["위도"], r["경도"]], radius=8,
        color="blue", fill=True, fill_opacity=0.8,
        popup=f"<b>기존 구르미</b><br>{r['터미널']}",
        tooltip=r["터미널"]
    ).add_to(m)

for _, r in df_dep_shuttle.iterrows():
    folium.CircleMarker(
        location=[r["위도"], r["경도"]], radius=8,
        color="red", fill=True, fill_opacity=0.8,
        popup=f"<b>출발지(셔틀이용자)</b><br>{r['정류장명']}<br>가중치: {r['기준_가중치']}",
        tooltip=r["정류장명"]
    ).add_to(m)

for _, r in df_dep_deadzone.iterrows():
    folium.CircleMarker(
        location=[r["위도"], r["경도"]], radius=8,
        color="orange", fill=True, fill_opacity=0.8,
        popup=f"<b>출발지(데드존이용자)</b><br>{r['정류장명']}<br>근처건물: {r['근처건물수']}개",
        tooltip=r["정류장명"]
    ).add_to(m)

for _, r in df_arr_station.iterrows():
    folium.CircleMarker(
        location=[r["위도"], r["경도"]], radius=8,
        color="green", fill=True, fill_opacity=0.8,
        popup=f"<b>도착지(역연계)</b><br>{r['정류장명']}<br>기준: {r['기준']}",
        tooltip=r["정류장명"]
    ).add_to(m)

for p in new_purple_points:
    folium.CircleMarker(
        location=[p["lat"], p["lon"]], radius=8,
        color="purple", fill=True, fill_opacity=0.8,
        popup=f"<b>도착지(산단직접통근)</b><br>{p['name']}<br>현장 검증 확정 후보지",
        tooltip=p["name"]
    ).add_to(m)

for i, coord in enumerate(Shuttle_station):
    folium.CircleMarker(
        location=[coord[0], coord[1]], radius=5,
        color="black", fill=True, fill_opacity=0.8,
        popup=f"<b>신규 셔틀 정류장 {i+1}</b>",
        tooltip=f"신규 셔틀 정류장 {i+1}"
    ).add_to(m)

for _, r in stations.iterrows():
    folium.Marker(
        location=[r["위도"], r["경도"]],
        icon=folium.Icon(color="gray", icon="info-sign"),
        tooltip=r["이름"]
    ).add_to(m)

legend_html = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
            padding:10px 15px;border:2px solid #ccc;border-radius:8px;font-size:12px'>
<b>구르미 신규 거점 후보</b><br>
<span style='color:blue'>●</span> 기존 구르미 거점 (7개소)<br>
<span style='color:red'>●</span> 출발지 - 셔틀 이용자<br>
<span style='color:orange'>●</span> 출발지 - 데드존 이용자<br>
<span style='color:green'>●</span> 도착지 - 역 연계<br>
<span style='color:purple'>●</span> 도착지 - 산단 직접 통근 (현장 확정)<br>
<span style='color:black'>●</span> 신규 셔틀 정류장<br>
<span style='color:gray'>●</span> 영주역/풍기역<br>
<span style='color:green'>▣</span> 산단 지역
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
display(m)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[출발지 - 셔틀 이용자] 4개
       정류장명       위도        경도  기준_가중치
0   영광중학교건너  36.8272  128.6194   970.3
1      샛빛학원  36.8178  128.6276   930.0
2  센텀파크.피카소  36.8109  128.6078   744.0
3     영동삼거리  36.8104  128.6283   704.9

[출발지 - 데드존 이용자] 7개

[도착지 - 역 연계] 2개
  정류장명         위도          경도   기준
0  영주역  36.810486  128.624387  영주역
1  풍기역  36.873088  128.524149  풍기역

  기존 구르미 7개소         : 8.4%
  신규 후보지 추가 후       : 45.9%
  커버리지 증가             : +37.5%p
